# 02 - Xây dựng Đồ thị Heterogeneous

## Phát hiện Giao dịch Gian lận dựa trên Mạng Neural Đồ thị

Notebook này xây dựng đồ thị heterogeneous từ dữ liệu IEEE-CIS:
- Load và tiền xử lý dữ liệu
- Xây dựng entity indexers
- Tạo HeteroData (PyTorch Geometric)
- Build edges (txn→entity, txn→txn)
- Time-based train/val/test split
- Verify data integrity

In [1]:
import sys
sys.path.insert(0, '..')

from src.config import set_seed, DEVICE
from src.data_loader import IEEECISDataLoader
from src.graph_builder import HeteroGraphBuilder

set_seed(42)
print(f"Device: {DEVICE}")

c:\Users\Tuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\Tuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch_scatter\_version_cpu.pyd
  import torch_geometric.typing
c:\Users\Tuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\Tuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch_sparse\_version_cpu.pyd
  import torch_geometric.typing


Device: cpu


## 1. Load dữ liệu

In [2]:
loader = IEEECISDataLoader()
df = loader.load()


LOADING DATA
Loaded 590,540 transactions
Fraud rate: 3.50% (20,663 fraud transactions)
txn_index range: 0 to 590539
Columns: ['txn_index', 'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'DeviceInfo', 'DeviceType', 'id_30', 'id_31']
[load] completed in 0.47s


## 2. Xây dựng đồ thị Heterogeneous

In [3]:
builder = HeteroGraphBuilder(df)
data = builder.build()


BUILDING HETEROGENEOUS GRAPH
Transaction nodes: 590,540
Building transaction features...
  Features shape: (590540, 6)
  Labels shape: (590540,) (fraud: 20,663)
[build_features] completed in 0.11s

BUILDING ENTITY INDEXERS
  card1: 8,419 unique values
  card2: 500 unique values
  card3: 90 unique values
  card4: 4 unique values
  card5: 91 unique values
  card6: 4 unique values
  addr1: 171 unique values
  addr2: 47 unique values
  p_email: 59 unique values
  r_email: 60 unique values
  device: 1,149 unique values
  devtype: 2 unique values
  os: 74 unique values
  browser: 111 unique values

Adding entity nodes...
  Added 14 entity types

BUILDING EDGES
  ('txn', 'has_card1', 'card1'): 583,716 edges
  ('txn', 'has_card2', 'card2'): 581,607 edges
  ('txn', 'has_card3', 'card3'): 588,943 edges
  ('txn', 'has_card4', 'card4'): 588,963 edges
  ('txn', 'has_card5', 'card5'): 586,242 edges
  ('txn', 'has_card6', 'card6'): 588,969 edges
  ('txn', 'has_addr1', 'addr1'): 524,635 edges
  ('txn

## 3. Thông tin đồ thị

In [4]:
print("=" * 60)
print("GRAPH SUMMARY")
print("=" * 60)

print(f"\nNode types ({len(data.node_types)}):")
for ntype in data.node_types:
    print(f"  {ntype}: {data[ntype].num_nodes:,} nodes")

print(f"\nEdge types ({len(data.edge_types)}):")
for et in data.edge_types:
    num_edges = data[et].edge_index.shape[1]
    print(f"  {et}: {num_edges:,} edges")

print(f"\nTransaction features: {data['txn'].x.shape}")
print(f"Labels: {data['txn'].y.shape}")
print(f"Train mask: {data['txn'].train_mask.sum():,}")
print(f"Val mask: {data['txn'].val_mask.sum():,}")
print(f"Test mask: {data['txn'].test_mask.sum():,}")

GRAPH SUMMARY

Node types (15):
  txn: 590,540 nodes
  card1: 8,419 nodes
  card2: 500 nodes
  card3: 90 nodes
  card4: 4 nodes
  card5: 91 nodes
  card6: 4 nodes
  addr1: 171 nodes
  addr2: 47 nodes
  p_email: 59 nodes
  r_email: 60 nodes
  device: 1,149 nodes
  devtype: 2 nodes
  os: 74 nodes
  browser: 111 nodes

Edge types (29):
  ('txn', 'has_card1', 'card1'): 583,716 edges
  ('card1', 'rev_has_card1', 'txn'): 583,716 edges
  ('txn', 'has_card2', 'card2'): 581,607 edges
  ('card2', 'rev_has_card2', 'txn'): 581,607 edges
  ('txn', 'has_card3', 'card3'): 588,943 edges
  ('card3', 'rev_has_card3', 'txn'): 588,943 edges
  ('txn', 'has_card4', 'card4'): 588,963 edges
  ('card4', 'rev_has_card4', 'txn'): 588,963 edges
  ('txn', 'has_card5', 'card5'): 586,242 edges
  ('card5', 'rev_has_card5', 'txn'): 586,242 edges
  ('txn', 'has_card6', 'card6'): 588,969 edges
  ('card6', 'rev_has_card6', 'txn'): 588,969 edges
  ('txn', 'has_addr1', 'addr1'): 524,635 edges
  ('addr1', 'rev_has_addr1', '

## 4. Lưu entity indexers

In [5]:
builder.save_indexers('../output/entity_indexers.json')
print("Entity indexers saved.")

Saved indexers to ../output/entity_indexers.json
Entity indexers saved.


## 5. Metadata cho models

In [6]:
metadata = data.metadata()
print(f"Node types: {metadata[0]}")
print(f"Edge types: {len(metadata[1])} relations")
print(f"Input feature dim: {data['txn'].x.shape[1]}")

Node types: ['txn', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'p_email', 'r_email', 'device', 'devtype', 'os', 'browser']
Edge types: 29 relations
Input feature dim: 6
